In [1]:
import sys
sys.path.append('/host/d/Github')
import os
import numpy as np
import pandas as pd
import nibabel as nb
import matplotlib.pyplot as plt

import radiomics
from radiomics import (
    featureextractor,  # This module is used for interaction with pyradiomics
)

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegressionCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.feature_selection import RFECV, RFE, SequentialFeatureSelector
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

import Osteosarcoma.functions_collection as ff
import Osteosarcoma.Build_lists.Build_list as Build_list


### feature selection step 1: ICC calculation for radimoics features from reader 1 and reader 2

In [3]:
df_reader1 = pd.read_excel('/host/d/projects/Habitats/radiomics/whole_image/radiomics_measurements_normalized.xlsx')
df_reader2 = pd.read_excel('/host/d/projects/Habitats/radiomics/whole_image/radiomics_measurements_normalized_reader2.xlsx')

# we only keep the rows in df_reader1 that are also in df_reader2 based on Patient_set and Patient_index
df_reader1 = df_reader1[df_reader1['Patient_index'].isin(df_reader2['Patient_index']) & df_reader1['Patient_set'].isin(df_reader2['Patient_set'])]
print(f'Number of cases in reader 1 after matching: {len(df_reader1)}')

non_feature_cols = ['Patient_set','Patient_index', 'Image_filepath', 'Mask_filepath']
feature_cols = [col for col in df_reader1.columns if col not in non_feature_cols]

# we need to calculate the ICC for each feature between reader 1 and reader 2, if it's >0.75, we keep it
# calculate ICC, import packages 

icc_rows = []
for f in feature_cols:
    x = df_reader1[f].values
    y = df_reader2[f].values
    icc = ff.icc2_1(x, y)
    if icc<0.75:
        print('feature:', f, ' ICC:', icc)
    icc_rows.append({'Feature': f, 'ICC': icc})

# only keep features with ICC > 0.75
icc_threshold = 0.75
icc_df = pd.DataFrame(icc_rows)
icc_df['included'] = icc_df['ICC'] > icc_threshold
icc_df['icc_status'] = np.where(icc_df['included'], 'included', 'excluded')
selected_features = icc_df.loc[icc_df['included'], 'Feature'].tolist()
print('original number of features:', len(feature_cols))
print(f'Number of features with ICC > {icc_threshold}: {len(selected_features)}')

# dropped features
dropped_features = icc_df.loc[~icc_df['included'], 'Feature'].tolist()
# save dropped features to excel, file name: dropped_features.xlsx, sheet_name: 'inter_reader_icc'
dropped_df = pd.DataFrame({'dropped_feature': dropped_features})

radiomics_root = '/host/d/projects/Habitats/radiomics/whole_image'
dropped_features_path = os.path.join(radiomics_root, 'dropped_features.xlsx')
icc_feature_list_path = os.path.join(radiomics_root, 'icc_feature_list.xlsx')
icc_output_path = os.path.join(radiomics_root, 'radiomics_measurements_ICC.xlsx')

with pd.ExcelWriter(dropped_features_path, engine='openpyxl') as writer:
    dropped_df.to_excel(writer, sheet_name='inter_reader_icc', index=False)

# Save full ICC feature list for documentation and reproducibility.
icc_df = icc_df[['Feature', 'ICC', 'included', 'icc_status']]
icc_df.to_excel(icc_feature_list_path, index=False)

# now we create "df" for reader 1 with only ICC-selected features.
# PCC filtering must start from this ICC-filtered table, not from the full normalized table.
df_reader1 = pd.read_excel('/host/d/projects/Habitats/radiomics/whole_image/radiomics_measurements_normalized.xlsx')
df_reader1_selected = df_reader1[['Patient_set','Patient_index', 'Image_filepath', 'Mask_filepath'] + selected_features]
df_reader1_selected.to_excel(icc_output_path, index=False)
print('Shape of df_reader1_selected:', df_reader1_selected.shape)
print('Saved ICC feature list:', icc_feature_list_path)
print('Saved ICC-filtered radiomics table:', icc_output_path)


Number of cases in reader 1 after matching: 28
feature: original_firstorder_Minimum  ICC: 0.6550166533723074
feature: wavelet-LLH_glszm_LargeAreaHighGrayLevelEmphasis  ICC: 0.7365548262642875
feature: wavelet-LHL_glszm_GrayLevelNonUniformity  ICC: 0.59375758883169
feature: wavelet-LHH_glcm_Contrast  ICC: 0.6351233228396639
feature: wavelet-LHH_glcm_Correlation  ICC: 0.6922743761177825
feature: wavelet-LHH_glcm_DifferenceAverage  ICC: 0.6465919570355538
feature: wavelet-LHH_glcm_JointEnergy  ICC: 0.7457528494056832
feature: wavelet-LHH_glcm_Idm  ICC: 0.6528546222043364
feature: wavelet-LHH_glcm_Id  ICC: 0.6584085034668954
feature: wavelet-LHH_glcm_InverseVariance  ICC: 0.6851645037566315
feature: wavelet-LHH_glrlm_RunLengthNonUniformityNormalized  ICC: 0.6473073779249607
feature: wavelet-LHH_glrlm_RunPercentage  ICC: 0.6631584872977915
feature: wavelet-LHH_gldm_LargeDependenceEmphasis  ICC: 0.6475696644940316
feature: wavelet-HLH_glcm_ClusterShade  ICC: 0.717754775037739
feature: wavele

### feature selection step 2: PCC


In [2]:
# ============================================================
# Feature selection step 2: PCC filtering
# ============================================================

radiomics_root = '/host/d/projects/Habitats/radiomics/whole_image'
# PCC must be performed after ICC filtering. This file is generated by the ICC cell above.
normalized_path = os.path.join(radiomics_root, 'radiomics_measurements_ICC.xlsx')
pcc_output_path = os.path.join(radiomics_root, 'radiomics_measurements_PCC.xlsx')
pcc_filtered_list_path = os.path.join(radiomics_root, 'pcc_filtered_list.xlsx')

# If True: rerun PCC search on the current ICC-filtered normalized table and overwrite pcc_filtered_list.xlsx.
# If False: reuse pcc_filtered_list.xlsx and only apply that feature list to the current table.
# After changing the upstream ICC input, this should be True at least once.
rerun_pcc_search = True

pcc_threshold = 0.90
non_feature_cols = ["Patient_set", "Patient_index", "Image_filepath", "Mask_filepath"]

df = pd.read_excel(normalized_path)
feature_cols = [c for c in df.columns if c not in non_feature_cols]

print('ICC-filtered normalized radiomics table:', normalized_path)
print('Shape:', df.shape)
print('Total feature columns:', len(feature_cols))
print('rerun_pcc_search:', rerun_pcc_search)

if rerun_pcc_search:
    # ------------------------------------------------------------
    # Recalculate PCC feature filtering from current normalized data.
    # ------------------------------------------------------------
    X = df[feature_cols].copy()
    corr = X.corr(method='pearson').abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

    to_drop = [col for col in upper.columns if (upper[col] > pcc_threshold).any()]
    to_drop_set = set(to_drop)

    pcc_filtered_list_df = pd.DataFrame({
        'feature_name': feature_cols,
        'pcc_status': ['excluded' if f in to_drop_set else 'included' for f in feature_cols],
        'included': [f not in to_drop_set for f in feature_cols],
        'original_feature_order': list(range(len(feature_cols))),
    })

    included_features_tmp = [f for f in feature_cols if f not in to_drop_set]
    pcc_feature_order_map = {f: i for i, f in enumerate(included_features_tmp)}
    pcc_filtered_list_df['pcc_feature_order'] = [
        pcc_feature_order_map.get(f, pd.NA) for f in feature_cols
    ]

    pcc_filtered_list_df.to_excel(pcc_filtered_list_path, index=False)

    print(f'PCC threshold: {pcc_threshold}')
    print(f'Dropped due to PCC > {pcc_threshold}: {len(to_drop)}')
    print(f'Remaining: {len(included_features_tmp)}')
    print('Saved PCC filtered list:', pcc_filtered_list_path)

else:
    # ------------------------------------------------------------
    # Reuse existing PCC feature list.
    # This is useful when new cases are added but the feature set should remain fixed.
    # ------------------------------------------------------------
    if not os.path.isfile(pcc_filtered_list_path):
        raise FileNotFoundError(
            f'pcc_filtered_list.xlsx not found: {pcc_filtered_list_path}. '
            'Set rerun_pcc_search=True once to generate it.'
        )

    pcc_filtered_list_df = pd.read_excel(pcc_filtered_list_path)

    required_cols = ['feature_name', 'included']
    for col in required_cols:
        if col not in pcc_filtered_list_df.columns:
            raise KeyError(f'Missing required column {col} in {pcc_filtered_list_path}')

    # Normalize boolean-like included column.
    if pcc_filtered_list_df['included'].dtype != bool:
        pcc_filtered_list_df['included'] = (
            pcc_filtered_list_df['included']
            .astype(str)
            .str.strip()
            .str.lower()
            .isin(['true', '1', 'yes', 'included'])
        )

    print('Loaded existing PCC filtered list:', pcc_filtered_list_path)
    print('Included features in list:', int(pcc_filtered_list_df['included'].sum()))
    print('Excluded features in list:', int((~pcc_filtered_list_df['included']).sum()))

print('\nPCC filtered list status counts:')
if 'pcc_status' in pcc_filtered_list_df.columns:
    print(pcc_filtered_list_df['pcc_status'].value_counts().to_string())
else:
    print(pcc_filtered_list_df['included'].value_counts().to_string())

Normalized radiomics table: /host/d/projects/Habitats/radiomics/whole_image/radiomics_measurements_normalized.xlsx
Shape: (351, 1019)
Total feature columns: 1015
rerun_pcc_search: False
Loaded existing PCC filtered list: /host/d/projects/Habitats/radiomics/whole_image/pcc_filtered_list.xlsx
Included features in list: 282
Excluded features in list: 733

PCC filtered list status counts:
excluded    733
included    282


In [3]:
# ============================================================
# Apply PCC filtered feature list and save radiomics_measurements_PCC.xlsx
# ============================================================

included_features = (
    pcc_filtered_list_df.loc[pcc_filtered_list_df['included'], 'feature_name']
    .astype(str)
    .tolist()
)

missing_in_current = [f for f in included_features if f not in df.columns]
if len(missing_in_current) > 0:
    raise RuntimeError(
        'Some included PCC features are missing from the current normalized table. '
        f'Number missing: {len(missing_in_current)}. Examples: {missing_in_current[:10]}'
    )

# Keep the included features in the order recorded in pcc_filtered_list.xlsx.
df_pcc = pd.concat(
    [
        df[non_feature_cols].copy(),
        df[included_features].copy(),
    ],
    axis=1,
)

df_pcc.to_excel(pcc_output_path, index=False)

print('Saved PCC-filtered radiomics table:', pcc_output_path)
print('Output shape:', df_pcc.shape)
print('Number of included PCC features:', len(included_features))
print('\nCases by Patient_set:')
print(df_pcc['Patient_set'].value_counts().sort_index().to_string())

Saved PCC-filtered radiomics table: /host/d/projects/Habitats/radiomics/whole_image/radiomics_measurements_PCC.xlsx
Output shape: (351, 286)
Number of included PCC features: 282

Cases by Patient_set:
set_1     99
set_2    231
set_3     21
